<a href="https://colab.research.google.com/github/cbonnin88/mes_allocs-ETL/blob/main/A_B_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import polars as pl
import plotly.express as px
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from google.cloud import bigquery
from google.colab import auth

# **Authenticate**

In [2]:
auth.authenticate_user()
project_id = 'mes-allocs-analysis'
client = bigquery.Client(project=project_id)

In [3]:
product_query = """
  SELECT * FROM `mes-allocs-analysis.dbt_mes_allocs.fct_product_events`
"""

df_product = pl.from_pandas(client.query(product_query).to_dataframe())

In [4]:
df_product.head()

user_id,event_type,event_at,device_id,device_type,page_url,session_id,is_conversion,gender,user_status,subscription_plan,plan_price
str,str,"datetime[μs, UTC]",str,str,str,str,bool,str,str,str,f64
"""user_6914""","""application_submitted""",2026-01-01 00:00:08 UTC,"""dev_8929""","""Desktop""","""https://www.mes-allocs.fr/appl…","""174888""",true,"""Female""","""Full-time Employee""","""Free""",0.0
"""user_2843""","""application_submitted""",2026-01-01 00:21:16 UTC,"""dev_3260""","""Desktop""","""https://www.mes-allocs.fr/appl…","""255917""",true,"""Male""","""Job-Seeker""","""Free""",0.0
"""user_11909""","""application_submitted""",2026-01-01 00:38:00 UTC,"""dev_8491""","""Desktop""","""https://www.mes-allocs.fr/appl…","""808932""",true,"""Male""","""Job-Seeker""","""Free""",0.0
"""user_11734""","""application_submitted""",2026-01-01 00:47:21 UTC,"""dev_2603""","""Desktop""","""https://www.mes-allocs.fr/appl…","""611878""",true,"""Non-Binary""","""Job-Seeker""","""Basic""",7.99
"""user_13111""","""application_submitted""",2026-01-01 00:59:43 UTC,"""dev_4694""","""Desktop""","""https://www.mes-allocs.fr/appl…","""479611""",true,"""Female""","""Job-Seeker""","""Free""",0.0


# **A/B Testing**
**Scenario:** You want to know if Desktop users have a statistically significant higher conversion rate than Mobile users.

- **Null Hypothesis** ($H_0$): There is no difference in conversion rates between devices.
- **Alternative Hypothesis** ($H_1$): There is a significant difference.

In [5]:
# 1. Preparing the groups
desktop_conv = df_product.filter(pl.col('device_type') == 'Desktop')['is_conversion'].cast(pl.Int8).to_numpy()
mobile_conv = df_product.filter(pl.col('device_type') == 'Mobile')['is_conversion'].cast(pl.Int8).to_numpy()

In [8]:
# 2. Perform T-test
t_stat, p_value = stats.ttest_ind(desktop_conv,mobile_conv)

print(f'T-Statistic: {t_stat:.4f}')
print(f'P-Value: {p_value:.4f}')

if p_value < 0.05:
  print(f'Result: {p_value}, Statistically Significant. Reject the Null Hypothesis')
else:
  print(f'Result: {p_value}, NOT Significant. Fail to reject the Null Hypothesis')

T-Statistic: 2.3531
P-Value: 0.0186
Result: 0.01861977391932101, Statistically Significant. Reject the Null Hypothesis


**Scenario:** Predicting Churn. I'll define a "Churned" user as someone who has not performed an application_submitted event.

In [11]:
# Creating features per user

user_features = df_product.group_by('user_id').agg([
    pl.col('event_type').count().alias('total_activity'),
    pl.col('is_conversion').max().cast(pl.Int8).alias('target'), # 1 if applied, 0 if churned
    pl.col('plan_price').first().alias('plan_price'),
    (pl.col('event_type') == 'simulation_started').sum().alias('sim_starts')
])

In [12]:
# Preparing X and y
X = user_features.select(['total_activity','plan_price','sim_starts']).to_pandas()
y = user_features.select('target').to_pandas().values.ravel()

In [13]:
# Splitting
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
# Training the Model
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train,y_train)

RandomForestClassifier()

In [15]:
# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.4f}')

              precision    recall  f1-score   support

           0       0.25      0.01      0.02       120
           1       0.96      1.00      0.98      2880

    accuracy                           0.96      3000
   macro avg       0.61      0.50      0.50      3000
weighted avg       0.93      0.96      0.94      3000

ROC-AUC Score: 0.5281


# **Engagement Gap**

In [17]:
plot_df = user_features.to_pandas()

In [18]:
plot_df['Outcome'] = plot_df['target'].map({1: 'Converted',0: 'Churned'})

In [19]:
fig_users = px.box(
    plot_df,
    x='Outcome',
    y='total_activity',
    color='Outcome',
    points='all',
    title='User Activity: Converters vs. Churners',
    labels={'total_activity':'Number of Events'}
)

fig_users.update_layout(showlegend=False)
fig_users.show()

# **The "Aha! Moment" Correlation**

In [20]:
fig_corr = px.scatter(
    plot_df,
    x='total_activity',
    y='sim_starts',
    color='Outcome',
    trendline='ols',
    title='Correlation: Total Activity vs. Simulations Started',
    labels = {'sim_starts':'Simulations Started','total_activity':'Total Events'}
)

fig_corr.show()